In [3]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Pinecone
import pinecone
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.prompts import PromptTemplate
from langchain.llms import CTransformers


c:\Users\HP\anaconda3\envs\mchatbot\lib\site-packages\pinecone\data\index.py:1: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [4]:
PINECONE_API_KEY = "pcsk_3GiiyL_86MgohU7CLEUeooHpERZd93U4hZPpvLfPamEwhrm2qhtsEUmYdgLgG9iQpg2wxW"
PINECONE_API_ENV = "us-east-1"


In [5]:
#Extract data from the PDF
def load_pdf(data):
    loader = DirectoryLoader(data,
                    glob="*.pdf",
                    loader_cls=PyPDFLoader)
    
    documents = loader.load()

    return documents

In [6]:
extracted_data = load_pdf("../data/")

In [7]:
# extracted_data

In [8]:
#Create text chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)

    return text_chunks

In [9]:
text_chunks = text_split(extracted_data)
print("length of my chunk:", len(text_chunks))

length of my chunk: 5859


In [10]:
# text_chunks

In [11]:
#download embedding model
def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

In [12]:
embeddings = download_hugging_face_embeddings()

C:\Users\HP\AppData\Local\Temp\ipykernel_27468\1337643473.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [13]:
embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [14]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [15]:
# query_result

In [ ]:
import os
from langchain_pinecone import PineconeVectorStore

# 1. Ensure your API Key is set in the environment
# Reminder: Use the NEW key you generated after rotating the leaked one!
os.environ["PINECONE_API_KEY"] = "pcsk_3GiiyL_86MgohU7CLEUeooHpERZd93U4hZPpvLfPamEwhrm2qhtsEUmYdgLgG9iQpg2wxW"

index_name = "medical-chatbot"

# 2. Modern way to create the vector store from your text chunks
# If your 'text_chunks' are LangChain Document objects:
docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    index_name=index_name
)



print("Ingestion complete. Your medical data is now vectorized in Pinecone!")

Ingestion complete. Your medical data is now vectorized in Pinecone!


In [20]:
import os
from langchain_pinecone import PineconeVectorStore

# 1. Set the environment variable so LangChain can find it
# USE YOUR NEW, SECURE API KEY HERE
os.environ["PINECONE_API_KEY"] = "pcsk_3GiiyL_86MgohU7CLEUeooHpERZd93U4hZPpvLfPamEwhrm2qhtsEUmYdgLgG9iQpg2wxW"

# 2. Now you can load your index
docsearch = PineconeVectorStore.from_existing_index(
    index_name="medical-chatbot", 
    embedding=embeddings
)

# 3. Test your search
query = "What are Allergies"
docs = docsearch.similarity_search(query, k=3)

print("Result:", docs)

Result: [Document(metadata={'page': 135.0, 'source': '..\\data\\Medical_book.pdf'}, page_content='Purpose\nAllergy is a reaction of the immune system. Nor-\nmally, the immune system responds to foreign microor-\nganisms and particles, like pollen or dust, by producing\nspecific proteins called antibodies that are capable of\nbinding to identifying molecules, or antigens, on the\nforeign organisms. This reaction between antibody and\nantigen sets off a series of reactions designed to protect\nthe body from infection. Sometimes, this same series of'), Document(metadata={'page': 135.0, 'source': '..\\data\\Medical_book.pdf'}, page_content='Purpose\nAllergy is a reaction of the immune system. Nor-\nmally, the immune system responds to foreign microor-\nganisms and particles, like pollen or dust, by producing\nspecific proteins called antibodies that are capable of\nbinding to identifying molecules, or antigens, on the\nforeign organisms. This reaction between antibody and\nantigen sets off

In [21]:
prompt_template="""
Use the following pieces of information to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.

Context: {context}
Question: {question}

Only return the helpful answer below and nothing else.
Helpful answer:
"""

In [22]:
PROMPT=PromptTemplate(template=prompt_template, input_variables=["context", "question"])
chain_type_kwargs={"prompt": PROMPT}

In [23]:
llm=CTransformers(
    model=r"C:\Users\HP\Downloads\End-to-end-Medical-Chatbot-using-Llama2\model\llama-2-7b-chat.ggmlv3.q4_0 (1).bin",
    model_type="llama",
    config={'max_new_tokens':512, 'temperature':0.8}
)

In [25]:
# Create the retriever
retriever = docsearch.as_retriever(search_kwargs={'k': 2})

# Check if it's a valid object (it should NOT say 'BaseRetriever')
print(f"Retriever Type: {type(retriever)}")

# Test it directly
test_docs = retriever.get_relevant_documents("What are allergies?")
print(f"Successfully retrieved {len(test_docs)} documents.")

Retriever Type: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


C:\Users\HP\AppData\Local\Temp\ipykernel_27468\3039537351.py:8: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  test_docs = retriever.get_relevant_documents("What are allergies?")


Successfully retrieved 2 documents.


In [27]:
import sys

# 1. Setup the retriever from your vector store
retriever = docsearch.as_retriever(search_kwargs={'k': 2})

# 2. Re-initialize the QA chain correctly
# (Ensuring we use the chain_type_kwargs you defined for your prompt template)
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=retriever,
    return_source_documents=True, 
    chain_type_kwargs=chain_type_kwargs
)

# 3. Diagnostic Check (Optional but helpful for debugging)
print("-" * 30)
print(f"Retriever initialized: {type(retriever)}")
print("Chatbot is ready. Type 'exit' or 'quit' to stop.")
print("-" * 30)

# 4. The Interactive Loop
while True:
    user_input = input("Input Prompt: ").strip()
    
    # Handle exit commands
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Exiting medical chatbot. Stay healthy!")
        break
        
    if not user_input:
        continue

    try:
        # Use .invoke() which is the modern (2026) standard for LangChain
        result = qa.invoke({"query": user_input})
        
        print("\nResponse:", result["result"])
        
        # Optional: Print the source documents for medical transparency
        # print("\nSources:", [doc.metadata.get('source', 'Unknown') for doc in result["source_documents"]])
        print("-" * 30)
        
    except Exception as e:
        print(f"An error occurred: {e}")

------------------------------
Retriever initialized: <class 'langchain_core.vectorstores.base.VectorStoreRetriever'>
Chatbot is ready. Type 'exit' or 'quit' to stop.
------------------------------



Response: Acne is a common skin condition characterized by inflamed red bumps, pimples, or whiteheads on the face, back, chest, and other areas of the body. It occurs when the pores on the skin become clogged with dead skin cells, oil, and bacteria, leading to infection and inflammation. Acne can be caused by a variety of factors, including hormonal changes, genetics, environmental stressors, and certain medications. Treatment options for acne include topical creams and gels, oral antibiotics, and lifestyle changes such as regular exercise and a healthy diet.
------------------------------
Exiting medical chatbot. Stay healthy!
